In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import random
import numpy as np
import torch
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

In [9]:
import data
#data.download_pipeline() # 다운로드 필요시
df = data.preprocess_pipeline()

In [10]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 99149 entries, 0 to 99148
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   접수일자                 99149 non-null  datetime64[us]
 1   접수지역                 99149 non-null  category      
 2   배달지역                 99149 non-null  category      
 3   총중량_kg               99149 non-null  float64       
 4   연도                   99149 non-null  int32         
 5   월                    99149 non-null  int32         
 6   일                    99149 non-null  int32         
 7   요일                   99149 non-null  int32         
 8   공휴일여부                99149 non-null  int64         
 9   다음_공휴일까지_남은일수        99149 non-null  int64         
 10  지난_공휴일부터_지난일수        99149 non-null  int64         
 11  공휴일_직전_3일내           99149 non-null  int64         
 12  공휴일_직후_2일내           99149 non-null  int64         
 13  lag_1                99149 non-null  float

접수일자                   0
접수지역                   0
배달지역                   0
총중량_kg                 0
연도                     0
월                      0
일                      0
요일                     0
공휴일여부                  0
다음_공휴일까지_남은일수          0
지난_공휴일부터_지난일수          0
공휴일_직전_3일내             0
공휴일_직후_2일내             0
lag_1                  0
lag_2                  0
lag_7                  0
lag_14                 0
rolling_7_mean         0
rolling_7_std          0
rolling_14_mean        0
ratio_lag1_to_roll7    0
dtype: int64

In [11]:
df_train, df_test = data.split_timeseries_data(df)

📊 시계열 데이터셋 분할 완료
 전체 데이터 기간: 2025-01-20 ~ 2026-06-30
 Train 데이터   : 2025-01-20 ~ 2026-05-29 (93,077건)
 Test 데이터    : 2026-06-01 ~ 2026-06-30 (6,072건)


In [ ]:
# ⚠️ 수정: LGBM의 early stopping이 df_test를 직접 보면 테스트 성능이 낙관적으로 부풀려짐(누수).
# 그래서 훈련기간(df_train) 내부에서 검증셋을 별도로 떼어낸다. df_test는 이 시점 이후로
# '최종 평가' 딱 한 번에만 사용하고, 어떤 학습/튜닝 결정에도 관여시키지 않는다.
df_train2, df_valid = data.split_timeseries_data(df_train, test_days=30)

In [ ]:
import lgbm, models

# 학습: df_train2로 학습하고, df_valid로 early stopping 기준을 잡는다 (df_test는 안 씀)
model_lgbm = lgbm.train_lgbm(df_train2, df_valid)

# 검증셋 예측 (앙상블 가중치 탐색용) + 테스트셋 예측 (최종 평가용)
preds_lgbm_valid = lgbm.predict_lgbm(model_lgbm, df_valid)
preds_lgbm_test = lgbm.predict_lgbm(model_lgbm, df_test)

print("[LGBM] 검증셋 성능:")
models.evaluate_outputs(df_valid["총중량_kg"], preds_lgbm_valid)
print("[LGBM] 테스트셋 성능 (최종, 한 번만 확인):")
models.evaluate_outputs(df_test["총중량_kg"], preds_lgbm_test)

In [13]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 6072 entries, 0 to 6071
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   접수일자                 6072 non-null   datetime64[us]
 1   접수지역                 6072 non-null   category      
 2   배달지역                 6072 non-null   category      
 3   총중량_kg               6072 non-null   float64       
 4   연도                   6072 non-null   int32         
 5   월                    6072 non-null   int32         
 6   일                    6072 non-null   int32         
 7   요일                   6072 non-null   int32         
 8   공휴일여부                6072 non-null   int64         
 9   다음_공휴일까지_남은일수        6072 non-null   int64         
 10  지난_공휴일부터_지난일수        6072 non-null   int64         
 11  공휴일_직전_3일내           6072 non-null   int64         
 12  공휴일_직후_2일내           6072 non-null   int64         
 13  lag_1                6072 non-null   float64

In [ ]:
import lstm

# 1. 학습 (df_train2만 사용 — df_valid/df_test는 학습에 관여하지 않음)
model_lstm, label_encoders, scaler = lstm.train_lstm(df_train2, epochs=30)

# 2. 검증셋 예측 (앙상블 가중치 탐색용)
preds_lstm_valid, valid_target_indices = lstm.predict_lstm(
    model_lstm, df_valid, label_encoders, scaler
)
y_valid_matched = df_valid.loc[valid_target_indices, "총중량_kg"].values

# 3. 테스트셋 예측 (최종 평가용)
preds_lstm_test, test_target_indices = lstm.predict_lstm(
    model_lstm, df_test, label_encoders, scaler
)
y_test_matched = df_test.loc[test_target_indices, "총중량_kg"].values

print("[LSTM] 검증셋 성능:")
models.evaluate_outputs(y_valid_matched, preds_lstm_valid)
print("[LSTM] 테스트셋 성능 (최종, 한 번만 확인):")
models.evaluate_outputs(y_test_matched, preds_lstm_test)

In [ ]:
import pandas as pd

# LGBM 예측은 df_valid/df_test 전체 인덱스 기준이고, LSTM 예측은 SEQ_LEN(14)만큼
# 앞부분이 빠진 target_indices 기준이라 인덱스를 맞춰서 비교해야 함.

# 1. 검증셋에서 최적 앙상블 가중치 탐색 (테스트셋은 여기 전혀 관여 안 함)
preds_lgbm_valid_matched = pd.Series(preds_lgbm_valid, index=df_valid.index).loc[valid_target_indices].values

w_lstm, w_lgb = models.find_best_ensemble_weight(
    y_valid_matched, preds_lstm_valid, preds_lgbm_valid_matched
)

# 2. 탐색한 가중치를 테스트셋에 '그대로' 적용해서 최종 평가 (여기서 가중치를 다시 조정하면 또 누수)
preds_lgbm_test_matched = pd.Series(preds_lgbm_test, index=df_test.index).loc[test_target_indices].values
final_pred = w_lstm * preds_lstm_test + w_lgb * preds_lgbm_test_matched

print("[앙상블] 최종 테스트셋 성능:")
models.evaluate_outputs(y_test_matched, final_pred)
